# Balance de Calor Oceánico — Ejemplo Guiado

Este notebook demuestra el uso del módulo `toolbox/balance_calor.py` del LOFA
para calcular los componentes del balance de calor superficial oceánico.

Está basado en los apuntes del curso de Oceanografía Física de la Universidad Austral
de Chile (profesor José Garcés Vargas).

**Requisitos:**
```
pip install numpy matplotlib pandas
```

**Ecuación central:**

$$Q_T = Q_{SW} + Q_{LW} + Q_{lat} \cdot (1 + R) + Q_v$$

donde:
- $Q_{SW}$: radiación solar neta de onda corta
- $Q_{LW}$: radiación neta de onda larga
- $Q_{lat}$: calor latente por evaporación
- $R$: cociente de Bowen (Qsen/Qlat)
- $Q_v$: calor advectivo por corrientes

---
*Laboratorio de Oceanografía Física (LOFA) — Universidad Austral de Chile*

In [ ]:
import sys
import os

# Agregar el directorio raíz del repositorio al path para importar el toolbox
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from toolbox import balance_calor as bc

print('Módulo balance_calor cargado.')
print('Constantes:', f'RHO={bc.RHO}, CP={bc.CP}, L={bc.L}')

## 1. Cálculo del Balance Total (Qt)

Valores típicos para la costa de Valdivia en **verano austral** (enero):

| Variable | Valor | Descripción |
|---|---|---|
| Qsw | +220 W/m² | Radiación solar neta (ganancia) |
| Qlw | −60 W/m² | Radiación onda larga neta (pérdida) |
| Qlat | −90 W/m² | Calor latente por evaporación (pérdida) |
| R | 0.10 | Cociente de Bowen (bajo sobre el océano en verano) |
| Qv | 0 W/m² | Sin advección en este ejemplo |

In [ ]:
# Verano austral (enero)
qt_verano = bc.Qt(Qsw=220, Qlw=-60, Qlat=-90, R=0.10, Qv=0)
print(f'Qt (verano) = {qt_verano:.1f} W/m²')

# Invierno austral (julio)
qt_invierno = bc.Qt(Qsw=80, Qlw=-40, Qlat=-50, R=0.35, Qv=0)
print(f'Qt (invierno) = {qt_invierno:.1f} W/m²')

print()
print('En verano el océano GANA calor (Qt > 0) → se calienta.')
print('En invierno el océano PIERDE calor (Qt < 0) → se enfría.')

## 2. Cambio de Temperatura (ΔT)

¿Cuánto se calienta/enfría una capa de 50 m en 30 días con el balance de verano?

$$\Delta T = \frac{Q_T \cdot \Delta t}{\rho \cdot C_p \cdot \Delta z}$$

In [ ]:
dT_verano = bc.delta_T(Qt_val=qt_verano, dt_dias=30, dz=50)
print(f'ΔT en 30 días (capa 50 m, verano) = {dT_verano:.3f} °C')

dT_invierno = bc.delta_T(Qt_val=qt_invierno, dt_dias=30, dz=50)
print(f'ΔT en 30 días (capa 50 m, invierno) = {dT_invierno:.3f} °C')

## 3. Ciclo Estacional Completo

Calculamos el balance mes a mes con valores típicos para Valdivia,
desglosando cada componente.

In [ ]:
# Valores mensuales representativos para Valdivia (basados en ERA5)
meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

Qsw_m  = np.array([220, 200, 160, 110,  70,  50,  55,  80, 120, 170, 200, 220])
Qlw_m  = np.array([-60, -58, -55, -48, -40, -35, -35, -38, -44, -52, -57, -60])
Qlat_m = np.array([-90, -85, -75, -60, -50, -45, -45, -50, -60, -72, -82, -90])
R_m    = np.array([0.10, 0.10, 0.12, 0.18, 0.25, 0.35, 0.35, 0.28, 0.20, 0.15, 0.11, 0.10])

# Calcular componentes
Qsen_m = np.array([bc.Qsen(R, Qlat) for R, Qlat in zip(R_m, Qlat_m)])
Qt_m   = np.array([bc.Qt(Qsw, Qlw, Qlat, R) 
                   for Qsw, Qlw, Qlat, R in zip(Qsw_m, Qlw_m, Qlat_m, R_m)])

df = pd.DataFrame({
    'Qsw' : Qsw_m,
    'Qlw' : Qlw_m,
    'Qlat': Qlat_m,
    'Qsen': Qsen_m,
    'Qt'  : Qt_m,
}, index=meses)

print('Balance de calor mensual (W/m²):')
print(df.round(1).to_string())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

x = np.arange(len(meses))

# Panel superior: componentes individuales
axes[0].plot(x, Qsw_m,  color='#E07A30', linewidth=2, marker='o', markersize=5, label='Qsw (solar)')
axes[0].plot(x, Qlw_m,  color='#D62728', linewidth=2, marker='o', markersize=5, label='Qlw (onda larga)')
axes[0].plot(x, Qlat_m, color='#1F77B4', linewidth=2, marker='o', markersize=5, label='Qlat (latente)')
axes[0].plot(x, Qsen_m, color='#9467BD', linewidth=2, marker='o', markersize=5, label='Qsen (sensible)')
axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
axes[0].set_ylabel('Flujo de calor (W/m²)', fontweight='bold')
axes[0].set_title('Ciclo estacional del balance de calor — Costa de Valdivia', fontweight='bold')
axes[0].legend(loc='upper right', fontsize=9)
axes[0].grid(True, alpha=0.3)

# Panel inferior: balance total con relleno
axes[1].plot(x, Qt_m, color='black', linewidth=2.5, marker='o', markersize=6, label='Qt (balance total)')
axes[1].fill_between(x, 0, Qt_m, where=(Qt_m > 0), color='#D62728', alpha=0.3, label='Ganancia de calor')
axes[1].fill_between(x, 0, Qt_m, where=(Qt_m < 0), color='#1F77B4', alpha=0.3, label='Pérdida de calor')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_ylabel('Qt (W/m²)', fontweight='bold')
axes[1].set_xlabel('Mes', fontweight='bold')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)

axes[1].set_xticks(x)
axes[1].set_xticklabels(meses)

plt.tight_layout()
plt.show()

## 4. Radiación solar a diferentes profundidades

La luz solar se atenúa exponencialmente con la profundidad:

$$Q_{SW}(z) = Q_{SO} \cdot e^{-kz}$$

El coeficiente de extinción $k$ depende de la turbidez del agua.

In [ ]:
Qso = 220  # Insolación superficial en verano (W/m²)

# Perfiles de atenuación para distintos tipos de agua
profundidades = np.linspace(0, 100, 200)

tipos_agua = {
    'Agua clara oceánica (k=0.04)'   : 0.04,
    'Agua costera típica (k=0.15)'   : 0.15,
    'Agua muy turbia (k=0.40)'       : 0.40,
}

fig, ax = plt.subplots(figsize=(7, 8))

for label, k in tipos_agua.items():
    Qz = np.array([bc.Qsw_z(Qso, k, z) for z in profundidades])
    ax.plot(Qz, -profundidades, linewidth=2, label=label)

ax.axvline(Qso * 0.01, color='gray', linestyle=':', linewidth=1.5)
ax.text(Qso * 0.01 + 1, -5, '1% de luz superficial\n(zona eufótica)', fontsize=9)

ax.set_xlabel('Radiación solar Qsw (W/m²)', fontweight='bold')
ax.set_ylabel('Profundidad (m)', fontweight='bold')
ax.set_title('Atenuación de la radiación solar con la profundidad', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Funciones disponibles en `toolbox/balance_calor.py`

| Función | Ecuación | Descripción |
|---|---|---|
| `bc.Qt(Qsw, Qlw, Qlat, R, Qv)` | $Q_T = Q_{SW} + Q_{LW} + Q_{lat}(1+R) + Q_v$ | Balance total |
| `bc.dH(dT, dz)` | $\Delta H = \rho C_p \Delta T \Delta z$ | Cambio de calor en el volumen |
| `bc.delta_T(Qt, dt_dias, dz)` | $\Delta T = Q_T \Delta t / (\rho C_p \Delta z)$ | Variación de temperatura |
| `bc.Qsw_z(Qso, k, z)` | $Q_{SW}(z) = Q_{SO} e^{-kz}$ | Radiación a profundidad z |
| `bc.Qlat(E_cm_dia)` | $Q_{lat} = -L \cdot E$ | Calor latente desde evaporación |
| `bc.Qsen(R, Qlat)` | $Q_{sen} = R \cdot Q_{lat}$ | Calor sensible |
| `bc.Qv(V, dz, gradT)` | $Q_v = \rho C_p V \Delta z (\Delta T/\Delta N)$ | Calor advectivo |